# 3. Named numbers

**The question:** what does a token look like inside the model?

In a trained transformer a token is a vector of numbers nobody chose. Here each of the 76 numbers has a name and
one job.

In [1]:
import helpers
from helpers import load, named, row, stream

model = load('base64')
r = model.r
print(f"{'name':<22}{'numbers':>8}   meaning")
meaning = {
  'one': 'always 1', 'bits': "the token's value", 'is_char': 'a base64 character', 'is_byte': 'a byte', 'is_pad': 'the = sign',
  'is_none': 'NONE', 'is_end': 'END', 'is_dec': 'the DEC token', 'is_enc': 'the ENC token',
  'phase': 'position in the group of 8', 'mode_dec': 'decoding', 'mode_enc': 'encoding',
  'out_bits': 'the output being assembled', 'out_pad': 'write =', 'out_end': 'write END'}
for head in ('enc_first', 'enc_second', 'dec_first', 'dec_second'):
  meaning |= {f'{head}.bits': f'what head {head} fetched', f'{head}.pad': 'it fetched =', f'{head}.missing': 'it found nothing'}
for name, (start, width) in r.S.blocks.items():
  print(f'{name:<22}{width:>8}   {meaning[name]}')
print('total', r.S.size)

name                   numbers   meaning
one                          1   always 1
bits                         8   the token's value
is_char                      1   a base64 character
is_byte                      1   a byte
is_pad                       1   the = sign
is_none                      1   NONE
is_end                       1   END
is_dec                       1   the DEC token
is_enc                       1   the ENC token
phase                        8   position in the group of 8
mode_dec                     1   decoding
mode_enc                     1   encoding
dec_first.bits               8   what head dec_first fetched
dec_first.pad                1   it fetched =
dec_first.missing            1   it found nothing
dec_second.bits              8   what head dec_second fetched
dec_second.pad               1   it fetched =
dec_second.missing           1   it found nothing
enc_first.bits               8   what head enc_first fetched
enc_first.pad                1   it fetch

## 1. A token going in

The embedding table holds what a token is; the position adds its `phase`. Here are the numbers that are not 0 for
the byte `H`, which is token 1 of the stream. `H` is `0x48`, binary `01001000`, so bits 3 and 6 are on.

In [2]:
tokens = stream(model)
print(tokens[:4])
print(named(model, row(model, 'x48', 1)))

['ENC', 'x48', 'x65', 'x6c']
{'one': 1.0, 'bits[3]': 1.0, 'bits[6]': 1.0, 'is_byte': 1.0, 'phase[1]': 1.0}


For the character `S`, value 18, binary `010010`, at token 4:

In [3]:
print(named(model, row(model, 'S', 4)))

{'one': 1.0, 'bits[1]': 1.0, 'bits[4]': 1.0, 'is_char': 1.0, 'phase[4]': 1.0}


And the mode token at the start of a group:

In [4]:
print(named(model, row(model, 'ENC', 0)))

{'one': 1.0, 'is_enc': 1.0, 'phase[0]': 1.0}


Most of the 76 numbers are 0 when a token enters. The fetched and output blocks start empty: attention and the MLP
write into them, as the next chapters show. Reading a vector means reading its names, which is why every step of
this model can be checked.